In [4]:
from transformers import RobertaTokenizer, RobertaModel
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [3]:
# Step 1: Load the text from a .txt file
file_path = 'Wood_Corpus_REFINED.txt'

with open(file_path, 'r', encoding='windows-1252') as file:
    text = file.read()

In [4]:
print(text)

--- Heartwood ---
1. Heartwood differs notably from its sapwood counterpart in terms of its spectral characteristics, especially within the Visible and Near-Infrared (VNIR) wavelength range of 400 to 1000 nm. Its unique spectral signature is governed by its compositional intricacies and material properties, which offer striking contrast in hyperspectral imaging.

Specifically, in the VNIR region, heartwood exhibits a pronounced reflectance peak around 550 nm, driven primarily by its distinct composition and density. The presence of extractive chemicals often imparts a darker characteristic to heartwood, which leads to higher absorption features in the green-yellow region of the spectrum. This contrasts with sapwood's spectral signature, which generally presents higher reflectance levels across the VNIR spectrum due to its lighter color and lower density, except for the water absorption feature near 970 nm. 

Furthermore, different cellular structures in heartwood and sapwood produce su

In [5]:
# Step 2: Load pre-trained RoBERTa tokenizer and model
#tokenizer = RobertaTokenizer.from_pretrained('roberta-large')
#model = RobertaModel.from_pretrained('roberta-large')

# Load pre-trained BERT-large tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased')  # BERT tokenizer
model = BertModel.from_pretrained('bert-large-uncased')   

In [6]:
# Step 3: Tokenize the text
inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)

#see tokens:
input_ids = inputs['input_ids'][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("Tokens:", tokens)

Tokens: ['[CLS]', '-', '-', '-', 'heart', '##wood', '-', '-', '-', '1', '.', 'in', 'the', 'realm', 'of', 'hyper', '##sp', '##ect', '##ral', 'imaging', ',', 'heart', '##wood', 'demonstrates', 'unique', 'spectral', 'characteristics', 'that', 'significantly', 'differ', 'from', 'those', 'of', 'sap', '##wood', ',', 'specifically', 'in', 'the', 'visible', 'and', 'near', '-', 'infrared', '(', 'v', '##nir', ')', 'range', '(', '400', '–', '1000', 'nm', ')', '.', 'when', 'assessed', 'through', 'hyper', '##sp', '##ect', '##ral', 'analysis', ',', 'heart', '##wood', "'", 's', 'spectra', 'frequently', 'display', 'distinct', 'absorption', 'features', 'at', 'around', 'the', '450', '-', '650', 'nm', 'and', '900', '-', '1000', 'nm', 'ranges', ',', 'typically', 'caused', 'by', 'the', 'presence', 'of', 'extract', '##ive', 'compounds', 'like', 'tan', '##nin', '##s', 'and', 'oils', 'that', 'are', 'produced', 'as', 'the', 'tree', 'ages', 'from', 'sap', '##wood', 'into', 'heart', '##wood', '.', 'these', 'comp

In [7]:
# Step 4: Extract embeddings
# Get the tokenized input IDs and attention mask
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']

# Run the inputs through the model
with torch.no_grad():
    
    outputs = model(input_ids, attention_mask=attention_mask)
    # Get the last hidden states (embeddings for each token)
    embeddings = outputs.last_hidden_state

In [8]:
# List of words to extract embeddings for
words = ["heartwood", "sapwood"]

# Tokenize the individual words
words_ids = {word: tokenizer.encode(word, add_special_tokens=False) for word in words}

# Initialize a dictionary to store embeddings
embeddings_dict = {}

# Loop through each word in the list
for word, word_id in words_ids.items():
    
    # Find the indices of this word in the tokenized input
    indices = [i for i, id_ in enumerate(input_ids[0]) if id_ in word_id]
    
    # Extract embeddings for the specific word
    embedding = embeddings[0, indices, :].mean(dim=0) if indices else None
    
    # Store the embedding in the dictionary
    embeddings_dict[word] = embedding

# Output the embeddings
for word, embedding in embeddings_dict.items():
    
    print(f"Embedding for {word}: {embedding}")
    
heartwood_embedding = embeddings_dict["heartwood"]
sapwood_embedding = embeddings_dict["sapwood"]

Embedding for heartwood: tensor([-0.9397, -0.6869, -0.1163,  ...,  0.0381,  0.7448,  0.4009])
Embedding for sapwood: tensor([-0.6007, -0.6435, -0.0284,  ..., -0.0155,  0.7985,  0.6359])


In [9]:
print(heartwood_embedding.shape, sapwood_embedding.shape)

similarity = cosine_similarity(heartwood_embedding.unsqueeze(0), sapwood_embedding.unsqueeze(0))
print("Cosine Similarity:", similarity)

torch.Size([1024]) torch.Size([1024])
Cosine Similarity: [[0.951842]]


In [10]:
# Save the embeddings to a .npy file
np.save("Wood_Embeddings_BERT.npy", embeddings_dict)

In [11]:
#load the embeddings back to verify
loaded_embeddings = np.load("Wood_Embeddings_REFINED.npy", allow_pickle=True).item()

#print(loaded_embeddings.shape)
print(loaded_embeddings)

{'heartwood': tensor([ 0.4190,  0.5055, -0.2181,  ...,  0.1561,  0.3533,  0.4029]), 'sapwood': tensor([ 0.4121,  0.4795, -0.2250,  ...,  0.1627,  0.3482,  0.3859])}


In [ ]:
'''
# Assume embedding_1 is from my example and embedding_2 is from your example
diff = heartwood_embedding - embedding_word_1

# Check if all elements are zero (exact match)
is_exact = torch.equal(heartwood_embedding, embedding_word_1)

# Alternatively, check for approximate equality (handles floating-point precision issues)
is_approx_equal = torch.allclose(heartwood_embedding, embedding_word_1, atol=1e-6)

print("Are the embeddings exactly the same?", is_exact)
print("Are the embeddings approximately the same?", is_approx_equal)
'''